In [ ]:
import torch
from torch import nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import ortho_group
import math
import itertools

In [ ]:
def ccgp4(h):
    p1 = nmvec(h[:,1],h[:,0],h[:,3]) - nmvec(h[:,1],h[:,0],h[:,2])
    p2 = nmvec(h[:,3],h[:,2],h[:,1]) - nmvec(h[:,3],h[:,2],h[:,0])
    p3 = nmvec(h[:,2],h[:,0],h[:,3]) - nmvec(h[:,2],h[:,0],h[:,1])
    p4 = nmvec(h[:,3],h[:,1],h[:,2]) - nmvec(h[:,3],h[:,1],h[:,0])
    return np.min([p1,p2,p3,p4])

def nmvec(h1,h0, ht):
    w = (h1 - h0)
    b = (np.linalg.norm(h1)**2 - np.linalg.norm(h0)**2)/2
    return (np.dot(w,ht) + b)/np.linalg.norm(h1 - h0)
    
def binary_rep(y):
    y_num = np.zeros((y.shape[0],))
    yint = np.zeros((y.shape[0], y.shape[1]))
    yint[y>0] = 1
    yint[y<0] = 0
    for i in range(y.shape[1]):
        y_num = y_num + yint[:,i]*(2**i)
    return y_num.reshape((y.shape[0],1))

def ps_highD(X_train, y_train):

    pslist = np.zeros(y_train.shape[1])
    y_train = y_train - np.mean(y_train, axis=0)
    for i in range(y_train.shape[1]):
        y_p0 = y_train[y_train[:, i] >0, :]
        y_n0 = y_train[y_train[:, i] <0, :]
        y_p = np.delete(y_p0,[i], 1)
        y_n = np.delete(y_n0,[i], 1)
        x_p = X_train[y_train[:, i] > 0, :]
        x_n = X_train[y_train[:, i] < 0, :]
        y_nump = binary_rep(y_p)
        y_numn = binary_rep(y_n)
        x_ps = x_p[np.argsort(y_nump[:, 0])]
        x_ns = x_n[np.argsort(y_numn[:, 0])]
        x_diff = x_ps - x_ns

        # compute PS
        idx = np.arange(x_diff.shape[0])
        x_idx, y_idx = np.meshgrid(idx, idx)
        x_idx1 = x_idx[y_idx > x_idx]
        y_idx1 = y_idx[y_idx > x_idx]
        x_idxf = x_idx1.flatten()
        y_idxf = y_idx1.flatten()
        dot = np.einsum('ij,ij->j', x_diff[x_idxf].T, x_diff[y_idxf].T)
        norm1 = np.linalg.norm(x_diff[x_idxf].T, axis = 0)
        norm2 = np.linalg.norm(x_diff[y_idxf].T, axis = 0)
        ps = np.mean(dot/norm1/norm2)
        pslist[i] = ps
    return np.mean(pslist)

def ps_highD_collapse(X_train0, y_train0):
    y_train = np.array(list(itertools.product([0, 1], repeat = y_train0.shape[1])))*2-1
    X_train = np.zeros((y_train.shape[0], X_train0.shape[1]))
    for i in range(y_train.shape[0]):
        idx_y = np.all(y_train0 == y_train[i], axis = 1)
        X_train[i] = np.mean(X_train0[idx_y], axis = 0)
        
    pslist = np.zeros(y_train.shape[1])
    for i in range(y_train.shape[1]):
        y_p0 = y_train[y_train[:, i] >0, :]
        y_n0 = y_train[y_train[:, i] <0, :]
        y_p = np.delete(y_p0, [i], 1)
        y_n = np.delete(y_n0, [i], 1)
        x_p = X_train[y_train[:, i] > 0, :]
        x_n = X_train[y_train[:, i] < 0, :]
        y_nump = binary_rep(y_p)
        y_numn = binary_rep(y_n)
        x_ps = x_p[np.argsort(y_nump[:, 0])]
        x_ns = x_n[np.argsort(y_numn[:, 0])]
        x_diff = x_ps - x_ns

        # compute PS
        idx = np.arange(x_diff.shape[0])
        x_idx, y_idx = np.meshgrid(idx, idx)
        x_idx1 = x_idx[y_idx > x_idx]
        y_idx1 = y_idx[y_idx > x_idx]
        x_idxf = x_idx1.flatten()
        y_idxf = y_idx1.flatten()
        dot = np.einsum('ij,ij->j', x_diff[x_idxf].T, x_diff[y_idxf].T)
        norm1 = np.linalg.norm(x_diff[x_idxf].T, axis = 0)
        norm2 = np.linalg.norm(x_diff[y_idxf].T, axis = 0)
        ps = np.mean(dot/norm1/norm2)
        pslist[i] = ps
    return np.mean(pslist), pslist

def ThreePlot(epoch_num, loss_list, ps_list, ccgp_list):
    fig, axs = plt.subplots(1, 3, figsize=(8, 2))
    axs[0].plot(epoch_num, loss_list)
    axs[0].set_ylabel('Loss')
    axs[1].plot(epoch_num, ps_list)
    axs[1].set_ylabel('PS')
    #axs[1].set_ylim([0,1.1])
    axs[2].plot(epoch_num, ccgp_list)
    axs[2].set_ylabel('CCGP (margin)')
    plt.subplots_adjust(wspace = 0.5)
    plt.show()

def PCAplot(pc_list, s_list):
    fig, axs = plt.subplots(1, 5, figsize=(8, 1))
    for fig_count in range(4):
        ax = axs[fig_count]
        Vrh = pc_list[fig_count]
        ax.scatter(Vrh[0, 0], Vrh[1, 0], color = 'red')
        ax.scatter(Vrh[0, 1], Vrh[1, 1], color = 'red', marker='^')
        ax.scatter(Vrh[0, 2], Vrh[1, 2], color = 'blue')
        ax.scatter(Vrh[0, 3], Vrh[1, 3], color = 'blue', marker = "^")
        ax.set_aspect('equal', 'box')
        fig_count = fig_count + 1
    axs[4].scatter(np.arange(1, len(s_list)+1), s_list)
    axs[4].set_xlim([0, len(s_list)])
    plt.subplots_adjust(wspace=0.5)
    plt.show()

def fixalignData(c, lat_dim, cy, output_size, r):

    num_points = 2**output_size
    sigma_x0 = torch.randn(lat_dim, dtype = torch.float32)
    sigma_x = sigma_x0/torch.linalg.vector_norm(sigma_x0)*r
    
    Vxt = torch.from_numpy(ortho_group.rvs(dim = num_points).astype(np.float32)) # or generate a Gaussian random matrix and then do QR decomposition
    
    cx = (1-c)*torch.matmul(torch.matmul(Vxt[:lat_dim].T, torch.diag(sigma_x**2)), Vxt[:lat_dim]) + c*cy/np.trace(cy)*torch.sum(sigma_x**2)   #np.matmul(X_train.T, X_train)
    xy_align = np.trace(np.matmul(cx, cy))/np.trace(cx)/np.trace(cy)*output_size
    
    _, Sx, Vxt1 = torch.linalg.svd(cx)
    
    return torch.matmul(torch.diag(torch.sqrt(Sx)), Vxt1), xy_align

In [ ]:
nonlinear_fun = nn.Hardsigmoid()
class CustomActivation(nn.Module):
    def __init__(self):
        super(CustomActivation, self).__init__()
    
    def forward(self, x):
        return torch.where(x < 0, torch.zeros_like(x), 1-torch.exp(-x))

class SqrtActivation(nn.Module):
    def __init__(self):
        super(SqrtActivation, self).__init__()
    
    def forward(self, x):
        return torch.where(x < 0, torch.zeros_like(x), torch.sqrt(x+1) - 1)


class TwoLayerNonlinearNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, epsilon1, epsilon2):
        super(TwoLayerNonlinearNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size, bias = True)
        self.fc2 = nn.Linear(hidden_size, output_size, bias = False)
        self.activation = SqrtActivation()
        nn.init.uniform_(self.fc1.weight, -epsilon1, epsilon1)
        nn.init.uniform_(self.fc2.weight, -epsilon2, epsilon2)
        nn.init.uniform_(self.fc1.bias, -epsilon1, epsilon1)

    def forward(self, x):
        #r = torch.relu(self.fc1(x))
        #r = 6*nonlinear_fun(self.fc1(x)-3)
        r = self.activation(self.fc1(x))
        #r = torch.tanh(self.fc1(x))
        z = self.fc2(r)
        return z, r

class TwoLayerLinearNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, epsilon1, epsilon2):
        super(TwoLayerLinearNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size, bias = True)
        self.fc2 = nn.Linear(hidden_size, output_size, bias = False)
        nn.init.uniform_(self.fc1.weight, -epsilon1, epsilon1)
        nn.init.uniform_(self.fc2.weight, -epsilon2, epsilon2)
        nn.init.uniform_(self.fc1.bias, -epsilon1, epsilon1)

    def forward(self, x):
        r = self.fc1(x)
        z = self.fc2(r)
        return z, r


class TwoLayerNonlinearNN_MuP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, epsilon1, epsilon2):
        super(TwoLayerNonlinearNN_MuP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size, bias = False)
        self.fc2 = nn.Linear(hidden_size, output_size, bias = False)
        #self.activation = SqrtActivation()
        nn.init.uniform_(self.fc1.weight, -epsilon1, epsilon1)
        nn.init.uniform_(self.fc2.weight, -epsilon2, epsilon2)
        #nn.init.uniform_(self.fc2.bias, -epsilon2, epsilon2)

    def forward(self, x):
        #r = torch.pow(torch.relu(self.fc1(x)), 3)
        #r = self.fc1(x)
        #r = torch.tanh(self.fc1(x))
        r = torch.relu(self.fc1(x))
        #r = 2*nonlinear_fun(self.fc1(x)-3)
        #r = self.activation(self.fc1(x))
        z = self.fc2(r)#/hidden_size
        return z, r

class TwoLayerLinearNN_MuP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, epsilon1, epsilon2):
        super(TwoLayerLinearNN_MuP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size, bias = False)
        self.fc2 = nn.Linear(hidden_size, output_size, bias = False)
        #self.activation = CustomActivation()
        nn.init.uniform_(self.fc1.weight, -epsilon1, epsilon1)
        nn.init.uniform_(self.fc2.weight, -epsilon2, epsilon2)

    def forward(self, x):
        #r = torch.pow(torch.relu(self.fc1(x)), 3)
        r = self.fc1(x)
        #r = torch.relu(self.fc1(x))
        #r = 2*nonlinear_fun(self.fc1(x)-3)
        #r = self.activation(self.fc1(x))
        z = self.fc2(r)#/hidden_size
        return z, r

class NLayerNonlinearNN_MuP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, epsilon1, epsilon2):
        super(NLayerNonlinearNN_MuP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size, bias = False)
        self.fc2 = nn.Linear(hidden_size, hidden_size, bias = False)
        #self.fc3 = nn.Linear(hidden_size, hidden_size, bias = False)
        #self.fc4 = nn.Linear(hidden_size, hidden_size, bias = False)
        self.fc5 = nn.Linear(hidden_size, output_size, bias = False)
        self.hiddensize = hidden_size
        nn.init.uniform_(self.fc1.weight, -epsilon1, epsilon1)
        nn.init.uniform_(self.fc2.weight, -epsilon2, epsilon2)
        #nn.init.uniform_(self.fc3.weight, -epsilon2, epsilon2)
        #nn.init.uniform_(self.fc4.weight, -epsilon2, epsilon2)
        nn.init.uniform_(self.fc5.weight, -epsilon2, epsilon2)

    def forward(self, x):
        r = self.fc1(x)
        r2 = self.fc2(r)/math.sqrt(self.hiddensize)
        #r3 = self.fc3(r)/math.sqrt(hidden_size)
        #r4 = self.fc4(r)/math.sqrt(hidden_size)
        z = self.fc5(r2)/self.hiddensize
        return z, r, r2#, r3, r4

class TwoLayerHardSig(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, epsilon1, epsilon2):
        super(TwoLayerHardSig, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size, bias = True)
        self.fc2 = nn.Linear(hidden_size, output_size, bias = False)
        nn.init.uniform_(self.fc1.weight, -epsilon1, epsilon1)
        nn.init.uniform_(self.fc2.weight, -epsilon2, epsilon2)
        nn.init.uniform_(self.fc1.bias, -epsilon1, epsilon1)

    def forward(self, x):
        r = 2*nonlinear_fun(self.fc1(x)-3)
        z = self.fc2(r)
        return z, r


class TwoLayerExp(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, epsilon1, epsilon2):
        super(TwoLayerExp, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size, bias = True)
        self.fc2 = nn.Linear(hidden_size, output_size, bias = False)
        self.activation = CustomActivation()
        nn.init.uniform_(self.fc1.weight, -epsilon1, epsilon1)
        nn.init.uniform_(self.fc2.weight, -epsilon2, epsilon2)
        nn.init.uniform_(self.fc1.bias, -epsilon1, epsilon1)

    def forward(self, x):
        r = self.activation(self.fc1(x))
        z = self.fc2(r)
        return z, r

class TwoLayerSqrt(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, epsilon1, epsilon2):
        super(TwoLayerSqrt, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size, bias = True)
        self.fc2 = nn.Linear(hidden_size, output_size, bias = False)
        self.activation = SqrtActivation()
        nn.init.uniform_(self.fc1.weight, -epsilon1, epsilon1)
        nn.init.uniform_(self.fc2.weight, -epsilon2, epsilon2)
        nn.init.uniform_(self.fc1.bias, -epsilon1, epsilon1)

    def forward(self, x):
        r = self.activation(self.fc1(x))
        z = self.fc2(r)
        return z, r



class TwoLayerTanh(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, epsilon1, epsilon2):
        super(TwoLayerTanh, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size, bias = True)
        self.fc2 = nn.Linear(hidden_size, output_size, bias = False)
        nn.init.uniform_(self.fc1.weight, -epsilon1, epsilon1)
        nn.init.uniform_(self.fc2.weight, -epsilon2, epsilon2)
        nn.init.uniform_(self.fc1.bias, -epsilon1, epsilon1)

    def forward(self, x):
        r = torch.tanh(2*self.fc1(x))
        z = self.fc2(r)
        return z, r

class TwoLayerHardTanh(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, epsilon1, epsilon2):
        super(TwoLayerHardTanh, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size, bias = True)
        self.fc2 = nn.Linear(hidden_size, output_size, bias = False)
        self.activation = nn.Hardtanh()
        nn.init.uniform_(self.fc1.weight, -epsilon1, epsilon1)
        nn.init.uniform_(self.fc2.weight, -epsilon2, epsilon2)
        nn.init.uniform_(self.fc1.bias, -epsilon1, epsilon1)

    def forward(self, x):
        r = 2*self.activation(self.fc1(x))
        z = self.fc2(r)
        return z, r


In [ ]:
def train_twolayerNN(X_train0, y_train, input_size, hidden_size, output_size, epsilon1, epsilon2, lr, num_epochs, print_idx, wd, noise):
    model = TwoLayerNonlinearNN(input_size, hidden_size, output_size, epsilon1, epsilon2)

    # Define the loss function (mean squared error) and optimizer (SGD)
    criterion = nn.MSELoss()

    optimizer = optim.SGD(model.parameters(), lr = lr,weight_decay = wd)

    epoch_num = np.zeros((100,))
    loss_list = np.zeros((100,))
    ps_list = np.zeros((100,))
    ccgp_list = np.zeros((100,))
    pr_list = np.zeros((100, 2))
    pc_list = np.zeros((5, X_train0.size(0), X_train0.size(0)))
    s_list = np.zeros((5, X_train0.size(0)))
    h_list = np.zeros((5, hidden_size, X_train0.size(0)))

    # training
    idx = 0
    idx_s = 0
    for epoch in range(num_epochs):
        X_train = X_train0 + noise*(torch.rand(X_train0.size(0), input_size) - 0.5)
        optimizer.zero_grad()
        model.train()
        pred, _ = model(X_train)
        loss = criterion(pred, y_train) #+0.15/400*torch.norm(x, 1)
        loss.backward()
        optimizer.step()

        if (epoch+1) % (num_epochs//100) == 0:
            epoch_num[idx] = epoch
            model.eval()
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            #r1 = torch.relu(torch.mm(w1, X_train.T) + biasl)
            r1 = torch.tanh(torch.mm(w1, X_train.T) + biasl)
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            loss_list[idx] = loss.item()
            r1n = r1.numpy() - r1mean.numpy()
            c1 = np.matmul(r1n.T, r1n)
            ps_list[idx] = ps_highD(r1n.T, y_train.numpy())
            #ccgp_list[idx] = ccgp4(r1n)
            pr_list[idx, 0] = np.trace(c1)**2/np.linalg.norm(c1)**2
            idx = idx + 1

        if ((epoch+1) % (num_epochs//4) == 0) or (epoch == 0):
            model.eval()
            if print_idx == 1:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            r1 = torch.relu(torch.mm(w1, X_train.T) + biasl)
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            #Ur, Sr, pc = torch.linalg.svd(r1-r1mean, full_matrices=False)
            #Vrh = torch.mm(torch.diag(Sr),pc)
            #pc_list[idx_s,:,:] = Vrh[:,:].numpy()
            #s_list[idx_s,:] = Sr.numpy()
            h_list[idx_s,:,:] = r1.numpy()
            idx_s = idx_s + 1

    return epoch_num, loss_list, ps_list, ccgp_list, pr_list, pc_list,  s_list, h_list

In [64]:
def train_twolayerNN_linear(X_train0, y_train, input_size, hidden_size, output_size, epsilon1, epsilon2, lr, num_epochs, print_idx, wd, noise):
    model = TwoLayerLinearNN(input_size, hidden_size, output_size, epsilon1, epsilon2)

    # Define the loss function (mean squared error) and optimizer (SGD)
    criterion = nn.MSELoss()

    optimizer = optim.SGD(model.parameters(), lr = lr, weight_decay = wd)

    epoch_num = np.zeros((100,))
    loss_list = np.zeros((100,))
    ps_list = np.zeros((100,))
    ccgp_list = np.zeros((100,))
    pr_list = np.zeros((100, 2))
    pc_list = np.zeros((5, X_train0.size(0), X_train0.size(0)))
    s_list = np.zeros((5, X_train0.size(0)))
    h_list = np.zeros((5, hidden_size, X_train0.size(0)))

    # training
    idx = 0
    idx_s = 0
    for epoch in range(num_epochs):
        X_train = X_train0 + noise*(torch.rand(X_train0.size(0), input_size) - 0.5)
        optimizer.zero_grad()
        model.train()
        pred, r = model(X_train)
        loss = criterion(pred, y_train) #+0.15/400*torch.norm(x, 1)
        loss.backward()
        optimizer.step()

        if (epoch+1) % (num_epochs//100) == 0:
            epoch_num[idx] = epoch
            model.eval()
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            r1 = r.detach().T
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            loss_list[idx] = loss.item()
            r1n = r1.numpy() - r1mean.numpy()
            c1 = np.matmul(r1n.T, r1n)
            ps_list[idx] = ps_highD(r1n.T, y_train.numpy())
            ccgp_list[idx] = ccgp4(r1n)
            pr_list[idx, 0] = np.trace(c1)**2/np.linalg.norm(c1)**2
            idx = idx + 1

        if ((epoch+1) % (num_epochs//4) == 0) or (epoch == 0):
            model.eval()
            if print_idx == 1:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            r1 = r.detach().T
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            Ur, Sr, pc = torch.linalg.svd(r1-r1mean, full_matrices=False)
            Vrh = torch.mm(torch.diag(Sr),pc)
            pc_list[idx_s,:,:] = Vrh[:,:].numpy()
            s_list[idx_s,:] = Sr.numpy()
            h_list[idx_s,:,:] = r1.numpy()
            idx_s = idx_s + 1

    return epoch_num, loss_list, ps_list, ccgp_list, pr_list, pc_list,  s_list, h_list

In [40]:
def train_twolayerNN_MuP(X_train0, y_train, input_size, hidden_size, output_size, epsilon1, epsilon2, lr, num_epochs, print_idx, wd, noise):
    model = TwoLayerNonlinearNN_MuP(input_size, hidden_size, output_size, epsilon1, epsilon2)

    # Define the loss function (mean squared error) and optimizer (SGD)
    criterion = nn.MSELoss()

    optimizer = optim.SGD(model.parameters(), lr = lr, weight_decay = wd)

    epoch_num = np.zeros((100,))
    loss_list = np.zeros((100,))
    ps_list = np.zeros((100,))
    ccgp_list = np.zeros((100,))
    pr_list = np.zeros((100, 2))
    pc_list = np.zeros((5, X_train0.size(0), X_train0.size(0)))
    s_list = np.zeros((5, X_train0.size(0)))
    h_list = np.zeros((5, hidden_size, X_train0.size(0)))

    # training
    idx = 0
    idx_s = 0
    for epoch in range(num_epochs):
        X_train = X_train0 + noise*(torch.rand(X_train0.size(0), input_size) - 0.5)
        optimizer.zero_grad()
        model.train()
        pred, r = model(X_train)
        loss = criterion(pred, y_train) #+0.15/400*torch.norm(x, 1)
        loss.backward()
        optimizer.step()

        if (epoch+1) % (num_epochs//100) == 0:
            epoch_num[idx] = epoch
            model.eval()
            #w1 = model.fc1.weight.data
            #r1 = torch.relu(torch.mm(w1, X_train.T))
            r1 = r.detach().T
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            loss_list[idx] = loss.item()
            r1n = r1.numpy() - r1mean.numpy()
            c1 = np.matmul(r1n.T, r1n)
            ps_list[idx] = ps_highD(r1n.T, y_train.numpy())
            ccgp_list[idx] = ccgp4(r1n)
            pr_list[idx, 0] = np.trace(c1)**2/np.linalg.norm(c1)**2
            idx = idx + 1

        if ((epoch+1) % (num_epochs//4) == 0) or (epoch == 0):
            model.eval()
            if print_idx == 1:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
            #w1 = model.fc1.weight.data
            #r1 = torch.relu(torch.mm(w1, X_train.T))
            r1 = r.detach().T
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            Ur, Sr, pc = torch.linalg.svd(r1-r1mean, full_matrices=False)
            Vrh = torch.mm(torch.diag(Sr),pc)
            pc_list[idx_s,:,:] = Vrh[:,:].numpy()
            s_list[idx_s,:] = Sr.numpy()
            h_list[idx_s,:,:] = r1.numpy()
            idx_s = idx_s + 1

    return epoch_num, loss_list, ps_list, ccgp_list, pr_list, pc_list,  s_list, h_list

In [1]:
def train_twolayerNN_MuP_meanfield(X_train0, y_train, input_size, hidden_size, output_size, epsilon1, epsilon2, lr, num_epochs, print_idx, wd, noise):
    model = TwoLayerNonlinearNN_MuP(input_size, hidden_size, output_size, epsilon1, epsilon2)

    # Define the loss function (mean squared error) and optimizer (SGD)
    criterion = nn.MSELoss()

    optimizer = optim.SGD([{'params': model.fc1.parameters(), 'weight_decay': wd},{'params':  model.fc2.parameters(), 'weight_decay': wd}], lr=lr)

    epoch_num = np.zeros((100,))
    loss_list = np.zeros((100,))
    ps_list = np.zeros((100,))
    ccgp_list = np.zeros((100,))
    pr_list = np.zeros((100, 2))
    pc_list = np.zeros((5, X_train0.size(0), X_train0.size(0)))
    s_list = np.zeros((5, X_train0.size(0)))
    h_list = np.zeros((5, hidden_size, X_train0.size(0)))

    # training
    idx = 0
    idx_s = 0
    for epoch in range(num_epochs):
        X_train = X_train0 + noise*(torch.rand(X_train0.size(0), input_size) - 0.5)
        optimizer.zero_grad()
        model.train()
        pred, _ = model(X_train)
        loss = criterion(pred, y_train) #+0.15/400*torch.norm(x, 1)
        loss.backward()
        optimizer.step()

        if (epoch+1) % (num_epochs//100) == 0:
            epoch_num[idx] = epoch
            model.eval()
            w1 = model.fc1.weight.data
            r1 = torch.relu(torch.mm(w1, X_train.T))
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            loss_list[idx] = loss.item()
            r1n = r1.numpy() - r1mean.numpy()
            c1 = np.matmul(r1n.T, r1n)
            ps_list[idx] = ps_highD(r1n.T, y_train.numpy())
            ccgp_list[idx] = ccgp4(r1n)
            pr_list[idx, 0] = np.trace(c1)**2/np.linalg.norm(c1)**2
            idx = idx + 1

        if ((epoch+1) % (num_epochs//4) == 0) or (epoch == 0):
            model.eval()
            if print_idx == 1:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
            w1 = model.fc1.weight.data
            r1 = torch.relu(torch.mm(w1, X_train.T))
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            Ur, Sr, pc = torch.linalg.svd(r1-r1mean, full_matrices=False)
            Vrh = torch.mm(torch.diag(Sr),pc)
            pc_list[idx_s,:,:] = Vrh[:,:].numpy()
            s_list[idx_s,:] = Sr.numpy()
            h_list[idx_s,:,:] = r1.numpy()
            idx_s = idx_s + 1

    return epoch_num, loss_list, ps_list, ccgp_list, pr_list, pc_list,  s_list, h_list

In [ ]:
def train_NlayerNN_MuP_meanfield(X_train0, y_train, input_size, hidden_size, output_size, epsilon1, epsilon2, lr, num_epochs, print_idx, wd, noise):
    model = NLayerNonlinearNN_MuP(input_size, hidden_size, output_size, epsilon1, epsilon2)

    # Define the loss function (mean squared error) and optimizer (SGD)
    criterion = nn.MSELoss()
    optimizer = optim.SGD([{'params': model.fc1.parameters(), 'weight_decay': wd},{'params':  model.fc2.parameters(), 'weight_decay': wd},{'params':  model.fc5.parameters(), 'weight_decay': wd}], lr=lr)

    epoch_num = np.zeros((100,))
    loss_list = np.zeros((100,))
    ps_list = np.zeros((100,2))
    ccgp_list = np.zeros((100,))
    pr_list = np.zeros((100, 2))
    pc_list = np.zeros((5, X_train0.size(0), X_train0.size(0)))
    s_list = np.zeros((5, X_train0.size(0)))
    h_list = np.zeros((5, hidden_size, X_train0.size(0)))
    h2_list = np.zeros((5, hidden_size, X_train0.size(0)))

    # training
    idx = 0
    idx_s = 0
    for epoch in range(num_epochs):
        X_train = X_train0 + noise*(torch.rand(X_train0.size(0), input_size) - 0.5)
        optimizer.zero_grad()
        model.train()
        pred, r, r2 = model(X_train)
        loss = criterion(pred, y_train) #+0.15/400*torch.norm(x, 1)
        loss.backward()
        optimizer.step()

        if (epoch+1) % (num_epochs//100) == 0:
            epoch_num[idx] = epoch
            model.eval()
            pred, r, r2 = model(X_train)
            loss_list[idx] = loss.item()
            ps_list[idx, 0] = ps_highD(r.detach().numpy(), y_train.numpy())
            ps_list[idx, 1] = ps_highD(r2.detach().numpy(), y_train.numpy())
            #ps_list[idx, 2] = ps_highD(r3.detach().numpy(), y_train.numpy())
            #ps_list[idx, 3] = ps_highD(r4.detach().numpy(), y_train.numpy())
            idx = idx + 1

        if ((epoch+1) % (num_epochs//4) == 0) or (epoch == 0):
            model.eval()
            h_list[idx_s,:,:] = r.detach().T.numpy()
            h2_list[idx_s,:,:] = r2.detach().T.numpy()
            idx_s = idx_s + 1
            if print_idx == 1:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

    return epoch_num, loss_list, ps_list, ccgp_list, h_list, h2_list

In [64]:
def train_twolayerNN_linear_MuP(X_train0, y_train, input_size, hidden_size, output_size, epsilon1, epsilon2, lr, num_epochs, print_idx, wd, noise):
    model = TwoLayerLinearNN_MuP(input_size, hidden_size, output_size, epsilon1, epsilon2)

    # Define the loss function (mean squared error) and optimizer (SGD)
    criterion = nn.MSELoss()

    optimizer = optim.SGD(model.parameters(), lr = lr, weight_decay = wd)

    epoch_num = np.zeros((100,))
    loss_list = np.zeros((100,))
    ps_list = np.zeros((100,))
    ccgp_list = np.zeros((100,))
    pr_list = np.zeros((100, 2))
    pc_list = np.zeros((5, X_train0.size(0), X_train0.size(0)))
    s_list = np.zeros((5, X_train0.size(0)))
    h_list = np.zeros((5, hidden_size, X_train0.size(0)))

    # training
    idx = 0
    idx_s = 0
    for epoch in range(num_epochs):
        X_train = X_train0 + noise*(torch.rand(X_train0.size(0), input_size) - 0.5)
        optimizer.zero_grad()
        model.train()
        pred, r1 = model(X_train)
        loss = criterion(pred, y_train) #+0.15/400*torch.norm(x, 1)
        loss.backward()
        optimizer.step()

        if (epoch+1) % (num_epochs//100) == 0:
            epoch_num[idx] = epoch
            model.eval()
            #w1 = model.fc1.weight.data
            #r1 = torch.relu(torch.mm(w1, X_train.T))
            r1 = r.detach().T
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            loss_list[idx] = loss.item()
            r1n = r1.numpy() - r1mean.numpy()
            c1 = np.matmul(r1n.T, r1n)
            ps_list[idx] = ps_highD(r1n.T, y_train.numpy())
            ccgp_list[idx] = ccgp4(r1n)
            pr_list[idx, 0] = np.trace(c1)**2/np.linalg.norm(c1)**2
            idx = idx + 1

        if ((epoch+1) % (num_epochs//4) == 0) or (epoch == 0):
            model.eval()
            if print_idx == 1:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
            #w1 = model.fc1.weight.data
            #r1 = torch.relu(torch.mm(w1, X_train.T))
            r1 = r.detach().T
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            Ur, Sr, pc = torch.linalg.svd(r1-r1mean, full_matrices=False)
            Vrh = torch.mm(torch.diag(Sr),pc)
            pc_list[idx_s,:,:] = Vrh[:,:].numpy()
            s_list[idx_s,:] = Sr.numpy()
            h_list[idx_s,:,:] = r1.numpy()
            idx_s = idx_s + 1

    return epoch_num, loss_list, ps_list, ccgp_list, pr_list, pc_list,  s_list, h_list

In [ ]:
def train_twolayerNN_MuPW12(X_train0, y_train, input_size, hidden_size, output_size, epsilon1, epsilon2, lr, num_epochs, print_idx, wd, noise):
    model = TwoLayerNonlinearNN_MuP(input_size, hidden_size, output_size, epsilon1, epsilon2)

    # Define the loss function (mean squared error) and optimizer (SGD)
    criterion = nn.MSELoss()

    optimizer = optim.SGD(model.parameters(), lr = lr, weight_decay = wd)

    epoch_num = np.zeros((100,))
    loss_list = np.zeros((100,))
    ps_list = np.zeros((100,))
    ccgp_list = np.zeros((100,))
    pr_list = np.zeros((100, 2))
    w1_list = np.zeros((10, hidden_size, input_size))
    w2_list = np.zeros((10, output_size, hidden_size))
    pc_list = np.zeros((10, X_train0.size(0), X_train0.size(0)))
    s_list = np.zeros((10, X_train0.size(0)))
    h_list = np.zeros((10, hidden_size, X_train0.size(0)))

    # training
    idx = 0
    idx_s = 0
    for epoch in range(num_epochs):
        X_train = X_train0 + noise*(torch.rand(X_train0.size(0), input_size) - 0.5)
        optimizer.zero_grad()
        model.train()
        pred, r = model(X_train)
        loss = criterion(pred, y_train) #+0.15/400*torch.norm(x, 1)
        loss.backward()
        optimizer.step()

        if (epoch+1) % (num_epochs//100) == 0:
            epoch_num[idx] = epoch
            model.eval()
            #w1 = model.fc1.weight.data
            #r1 = torch.relu(torch.mm(w1, X_train.T))
            r1 = r.detach().T
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            loss_list[idx] = loss.item()
            r1n = r1.numpy() - r1mean.numpy()
            c1 = np.matmul(r1.numpy().T, r1.numpy())
            c1n = np.matmul(r1n.T, r1n)
            ps_list[idx],_ = ps_highD_collapse(r1n.T, y_train.numpy())
            ccgp_list[idx] = ccgp4(r1n)
            pr_list[idx, 0] = np.trace(c1n)**2/np.linalg.norm(c1n)**2
            pr_list[idx, 1] = np.trace(c1)**2/np.linalg.norm(c1)**2
            idx = idx + 1
            

        if ((epoch+1) % (num_epochs//9) == 0) or (epoch == 0):
            model.eval()
            if print_idx == 1:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
            w1 = model.fc1.weight.data
            r1 = r.detach().T
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            Ur, Sr, pc = torch.linalg.svd(r1-r1mean, full_matrices=False)
            Vrh = torch.mm(torch.diag(Sr),pc)
            pc_list[idx_s,:,:] = Vrh[:,:].numpy()
            s_list[idx_s,:] = Sr.numpy()
            h_list[idx_s,:,:] = r1.numpy()
            w1_list[idx_s,:,:] = w1.numpy()
            w2_list[idx_s,:,:] = model.fc2.weight.data.numpy()
            idx_s = idx_s + 1

    return epoch_num, loss_list, ps_list, ccgp_list, pr_list, pc_list,  s_list, h_list, w1_list, w2_list

In [2]:
def train_twolayerNN_MuPW12_SGD(X_train0, y_train, input_size, hidden_size, output_size, epsilon1, epsilon2, lr, bs, num_epochs, print_idx, wd, noise):
    model = TwoLayerNonlinearNN_MuP(input_size, hidden_size, output_size, epsilon1, epsilon2)

    # Define the loss function (mean squared error) and optimizer (SGD)
    criterion = nn.MSELoss()

    optimizer = optim.SGD(model.parameters(), lr = lr, weight_decay = wd)

    epoch_num = np.zeros((100,))
    loss_list = np.zeros((100,))
    ps_list = np.zeros((100,))
    ccgp_list = np.zeros((100,))
    pr_list = np.zeros((100, 2))
    w1_list = np.zeros((10, hidden_size, input_size))
    w2_list = np.zeros((10, output_size, hidden_size))
    pc_list = np.zeros((10, X_train0.size(0), X_train0.size(0)))
    s_list = np.zeros((10, X_train0.size(0)))
    h_list = np.zeros((10, hidden_size, X_train0.size(0)))

    # training
    idx = 0
    idx_s = 0
    for epoch in range(num_epochs):
        x_idx = np.random.choice(X_train0.size(0), bs, replace= False)
        X_train = X_train0[x_idx] + noise*(torch.rand(bs, input_size) - 0.5)
        optimizer.zero_grad()
        model.train()
        pred, r = model(X_train)
        loss = criterion(pred, y_train[x_idx]) #+0.15/400*torch.norm(x, 1)
        loss.backward()
        optimizer.step()

        if (epoch+1) % (num_epochs//100) == 0:
            epoch_num[idx] = epoch
            model.eval()
            #w1 = model.fc1.weight.data
            #r1 = torch.relu(torch.mm(w1, X_train.T))
            r1 = r.detach().T
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            loss_list[idx] = loss.item()
            r1n = r1.numpy() - r1mean.numpy()
            c1 = np.matmul(r1n.T, r1n)
            ps_list[idx] = ps_highD(r1n.T, y_train.numpy())
            ccgp_list[idx] = ccgp4(r1n)
            pr_list[idx, 0] = np.trace(c1)**2/np.linalg.norm(c1)**2
            idx = idx + 1
        
            

        if ((epoch+1) % (num_epochs//9) == 0) or (epoch == 0):
            model.eval()
            if print_idx == 1:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
            w1 = model.fc1.weight.data
            r1 = torch.relu(torch.mm(w1, X_train0.mT))
            #r1 = torch.tanh(torch.mm(w1, X_train.T))
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train0.size(0),dtype = torch.float32))
            Ur, Sr, pc = torch.linalg.svd(r1-r1mean, full_matrices=False)
            Vrh = torch.mm(torch.diag(Sr),pc)
            pc_list[idx_s,:,:] = Vrh[:,:].numpy()
            s_list[idx_s,:] = Sr.numpy()
            h_list[idx_s,:,:] = r1.numpy()
            w1_list[idx_s,:,:] = w1.numpy()
            w2_list[idx_s,:,:] = model.fc2.weight.data.numpy()
            idx_s = idx_s + 1

    return epoch_num, loss_list, ps_list, ccgp_list, pr_list, pc_list,  s_list, h_list, w1_list, w2_list

In [ ]:
def train_NlayerNN(X_train0, y_train, input_size, hidden_size, output_size, epsilon1, epsilon2, lr, num_epochs, print_idx, wd, noise):
    model = NLayerNonlinearNN(input_size, hidden_size, output_size, epsilon1, epsilon2)

    # Define the loss function (mean squared error) and optimizer (SGD)
    criterion = nn.MSELoss()

    optimizer = optim.SGD(model.parameters(), lr = lr, weight_decay = wd)

    epoch_num = np.zeros((100,))
    loss_list = np.zeros((100,))
    ps_list = np.zeros((100,))
    ccgp_list = np.zeros((100,))
    pr_list = np.zeros((100, 2))
    pc_list = np.zeros((5, X_train0.size(0), X_train0.size(0)))
    s_list = np.zeros((5, X_train0.size(0)))
    h_list = np.zeros((5, hidden_size, X_train0.size(0)))

    # training
    idx = 0
    idx_s = 0
    for epoch in range(num_epochs):
        X_train = X_train0 + noise*(torch.rand(X_train0.size(0), input_size) - 0.5)
        optimizer.zero_grad()
        model.train()
        pred, _ = model(X_train)
        loss = criterion(pred, y_train) #+0.15/400*torch.norm(x, 1)
        loss.backward()
        optimizer.step()

        if (epoch+1) % (num_epochs//100) == 0:
            epoch_num[idx] = epoch
            model.eval()
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            r1 = torch.relu(torch.mm(w1, X_train.T) + biasl)
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            loss_list[idx] = loss.item()
            r1n = r1.numpy() - r1mean.numpy()
            c1 = np.matmul(r1n.T, r1n)
            ps_list[idx] = ps_highD(r1n.T, y_train.numpy())
            ccgp_list[idx] = ccgp4(r1n)
            pr_list[idx, 0] = np.trace(c1)**2/np.linalg.norm(c1)**2
            idx = idx + 1

        if ((epoch+1) % (num_epochs//4) == 0) or (epoch == 0):
            model.eval()
            if print_idx == 1:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            r1 = torch.relu(torch.mm(w1, X_train.T) + biasl)
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            Ur, Sr, pc = torch.linalg.svd(r1-r1mean, full_matrices=False)
            Vrh = torch.mm(torch.diag(Sr),pc)
            pc_list[idx_s,:,:] = Vrh[:,:].numpy()
            s_list[idx_s,:] = Sr.numpy()
            h_list[idx_s,:,:] = r1.numpy()
            idx_s = idx_s + 1

    return epoch_num, loss_list, ps_list, ccgp_list, pr_list, pc_list,  s_list, h_list

In [63]:
def train_twolayerNN_Hardsig(X_train0, y_train, input_size, hidden_size, output_size, epsilon1, epsilon2, lr, num_epochs, print_idx, wd, noise):
    model = TwoLayerHardSig(input_size, hidden_size, output_size, epsilon1, epsilon2)

    # Define the loss function (mean squared error) and optimizer (SGD)
    criterion = nn.MSELoss()

    optimizer = optim.SGD(model.parameters(), lr = lr, momentum=0, weight_decay = wd)

    epoch_num = np.zeros((100,))
    loss_list = np.zeros((100,))
    ps_list = np.zeros((100,))
    ccgp_list = np.zeros((100,))
    pr_list = np.zeros((100, 2))
    pc_list = np.zeros((5, X_train0.size(0), X_train0.size(0)))
    s_list = np.zeros((5, X_train0.size(0)))
    h_list = np.zeros((5, hidden_size, X_train0.size(0)))

    # training
    idx = 0
    idx_s = 0
    for epoch in range(num_epochs):
        X_train = X_train0 + noise*(torch.rand(X_train0.size(0), input_size) - 0.5)
        optimizer.zero_grad()
        model.train()
        pred, _ = model(X_train)
        loss = criterion(pred, y_train) #+0.15/400*torch.norm(x, 1)
        loss.backward()
        optimizer.step()

        if (epoch+1) % (num_epochs//100) == 0:
            epoch_num[idx] = epoch
            model.eval()
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            #r1 = torch.relu(torch.mm(w1, X_train.T) + biasl)
            r1 = torch.tanh(torch.mm(w1, X_train.T) + biasl)
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            loss_list[idx] = loss.item()
            r1n = r1.numpy() - r1mean.numpy()
            c1 = np.matmul(r1n.T, r1n)
            ps_list[idx] = ps_highD(r1n.T, y_train.numpy())
            #ccgp_list[idx] = ccgp4(r1n)
            pr_list[idx, 0] = np.trace(c1)**2/np.linalg.norm(c1)**2
            idx = idx + 1

        if ((epoch+1) % (num_epochs//4) == 0) or (epoch == 0):
            model.eval()
            if print_idx == 1:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            r1 = torch.relu(torch.mm(w1, X_train.T) + biasl)
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            #Ur, Sr, pc = torch.linalg.svd(r1-r1mean, full_matrices=False)
            #Vrh = torch.mm(torch.diag(Sr),pc)
            #pc_list[idx_s,:,:] = Vrh[:,:].numpy()
            #s_list[idx_s,:] = Sr.numpy()
            h_list[idx_s,:,:] = r1.numpy()
            idx_s = idx_s + 1

    return epoch_num, loss_list, ps_list, ccgp_list, pr_list, pc_list,  s_list, h_list

In [63]:
def train_twolayerNN_Exp(X_train0, y_train, input_size, hidden_size, output_size, epsilon1, epsilon2, lr, num_epochs, print_idx, wd, noise):
    model = TwoLayerExp(input_size, hidden_size, output_size, epsilon1, epsilon2)

    # Define the loss function (mean squared error) and optimizer (SGD)
    criterion = nn.MSELoss()

    optimizer = optim.SGD(model.parameters(), lr = lr, momentum=0, weight_decay = wd)

    epoch_num = np.zeros((100,))
    loss_list = np.zeros((100,))
    ps_list = np.zeros((100,))
    ccgp_list = np.zeros((100,))
    pr_list = np.zeros((100, 2))
    pc_list = np.zeros((5, X_train0.size(0), X_train0.size(0)))
    s_list = np.zeros((5, X_train0.size(0)))
    h_list = np.zeros((5, hidden_size, X_train0.size(0)))

    # training
    idx = 0
    idx_s = 0
    for epoch in range(num_epochs):
        X_train = X_train0 + noise*(torch.rand(X_train0.size(0), input_size) - 0.5)
        optimizer.zero_grad()
        model.train()
        pred, _ = model(X_train)
        loss = criterion(pred, y_train) #+0.15/400*torch.norm(x, 1)
        loss.backward()
        optimizer.step()

        if (epoch+1) % (num_epochs//100) == 0:
            epoch_num[idx] = epoch
            model.eval()
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            #r1 = torch.relu(torch.mm(w1, X_train.T) + biasl)
            r1 = torch.tanh(torch.mm(w1, X_train.T) + biasl)
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            loss_list[idx] = loss.item()
            r1n = r1.numpy() - r1mean.numpy()
            c1 = np.matmul(r1n.T, r1n)
            ps_list[idx] = ps_highD(r1n.T, y_train.numpy())
            #ccgp_list[idx] = ccgp4(r1n)
            pr_list[idx, 0] = np.trace(c1)**2/np.linalg.norm(c1)**2
            idx = idx + 1

        if ((epoch+1) % (num_epochs//4) == 0) or (epoch == 0):
            model.eval()
            if print_idx == 1:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            r1 = torch.relu(torch.mm(w1, X_train.T) + biasl)
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            #Ur, Sr, pc = torch.linalg.svd(r1-r1mean, full_matrices=False)
            #Vrh = torch.mm(torch.diag(Sr),pc)
            #pc_list[idx_s,:,:] = Vrh[:,:].numpy()
            #s_list[idx_s,:] = Sr.numpy()
            h_list[idx_s,:,:] = r1.numpy()
            idx_s = idx_s + 1

    return epoch_num, loss_list, ps_list, ccgp_list, pr_list, pc_list,  s_list, h_list

In [63]:
def train_twolayerNN_Sqrt(X_train0, y_train, input_size, hidden_size, output_size, epsilon1, epsilon2, lr, num_epochs, print_idx, wd, noise):
    model = TwoLayerSqrt(input_size, hidden_size, output_size, epsilon1, epsilon2)

    # Define the loss function (mean squared error) and optimizer (SGD)
    criterion = nn.MSELoss()

    optimizer = optim.SGD(model.parameters(), lr = lr, momentum=0, weight_decay = wd)

    epoch_num = np.zeros((100,))
    loss_list = np.zeros((100,))
    ps_list = np.zeros((100,))
    ccgp_list = np.zeros((100,))
    pr_list = np.zeros((100, 2))
    pc_list = np.zeros((5, X_train0.size(0), X_train0.size(0)))
    s_list = np.zeros((5, X_train0.size(0)))
    h_list = np.zeros((5, hidden_size, X_train0.size(0)))

    # training
    idx = 0
    idx_s = 0
    for epoch in range(num_epochs):
        X_train = X_train0 + noise*(torch.rand(X_train0.size(0), input_size) - 0.5)
        optimizer.zero_grad()
        model.train()
        pred, _ = model(X_train)
        loss = criterion(pred, y_train) #+0.15/400*torch.norm(x, 1)
        loss.backward()
        optimizer.step()

        if (epoch+1) % (num_epochs//100) == 0:
            epoch_num[idx] = epoch
            model.eval()
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            #r1 = torch.relu(torch.mm(w1, X_train.T) + biasl)
            r1 = torch.tanh(torch.mm(w1, X_train.T) + biasl)
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            loss_list[idx] = loss.item()
            r1n = r1.numpy() - r1mean.numpy()
            c1 = np.matmul(r1n.T, r1n)
            ps_list[idx] = ps_highD(r1n.T, y_train.numpy())
            #ccgp_list[idx] = ccgp4(r1n)
            pr_list[idx, 0] = np.trace(c1)**2/np.linalg.norm(c1)**2
            idx = idx + 1

        if ((epoch+1) % (num_epochs//4) == 0) or (epoch == 0):
            model.eval()
            if print_idx == 1:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            r1 = torch.relu(torch.mm(w1, X_train.T) + biasl)
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            #Ur, Sr, pc = torch.linalg.svd(r1-r1mean, full_matrices=False)
            #Vrh = torch.mm(torch.diag(Sr),pc)
            #pc_list[idx_s,:,:] = Vrh[:,:].numpy()
            #s_list[idx_s,:] = Sr.numpy()
            h_list[idx_s,:,:] = r1.numpy()
            idx_s = idx_s + 1

    return epoch_num, loss_list, ps_list, ccgp_list, pr_list, pc_list,  s_list, h_list

In [63]:
def train_twolayerNN_Hardtanh(X_train0, y_train, input_size, hidden_size, output_size, epsilon1, epsilon2, lr, num_epochs, print_idx, wd, noise):
    model = TwoLayerHardTanh(input_size, hidden_size, output_size, epsilon1, epsilon2)

    # Define the loss function (mean squared error) and optimizer (SGD)
    criterion = nn.MSELoss()

    optimizer = optim.SGD(model.parameters(), lr = lr, momentum=0, weight_decay = wd)

    epoch_num = np.zeros((100,))
    loss_list = np.zeros((100,))
    ps_list = np.zeros((100,))
    ccgp_list = np.zeros((100,))
    pr_list = np.zeros((100, 2))
    pc_list = np.zeros((5, X_train0.size(0), X_train0.size(0)))
    s_list = np.zeros((5, X_train0.size(0)))
    h_list = np.zeros((5, hidden_size, X_train0.size(0)))

    # training
    idx = 0
    idx_s = 0
    for epoch in range(num_epochs):
        X_train = X_train0 + noise*(torch.rand(X_train0.size(0), input_size) - 0.5)
        optimizer.zero_grad()
        model.train()
        pred, r = model(X_train)
        loss = criterion(pred, y_train) #+0.15/400*torch.norm(x, 1)
        loss.backward()
        optimizer.step()

        if (epoch+1) % (num_epochs//100) == 0:
            epoch_num[idx] = epoch
            model.eval()
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            #r1 = torch.relu(torch.mm(w1, X_train.T) + biasl)
            r1 = r.detach().T
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            loss_list[idx] = loss.item()
            r1n = r1.numpy() - r1mean.numpy()
            c1 = np.matmul(r1n.T, r1n)
            ps_list[idx] = ps_highD(r1n.T, y_train.numpy())
            #ccgp_list[idx] = ccgp4(r1n)
            pr_list[idx, 0] = np.trace(c1)**2/np.linalg.norm(c1)**2
            idx = idx + 1

        if ((epoch+1) % (num_epochs//4) == 0) or (epoch == 0):
            model.eval()
            if print_idx == 1:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            r1 = r.detach().T
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            h_list[idx_s,:,:] = r1.numpy()
            idx_s = idx_s + 1

    return epoch_num, loss_list, ps_list, ccgp_list, pr_list, pc_list,  s_list, h_list

In [63]:
def train_twolayerNN_Tanh(X_train0, y_train, input_size, hidden_size, output_size, epsilon1, epsilon2, lr, num_epochs, print_idx, wd, noise):
    model = TwoLayerTanh(input_size, hidden_size, output_size, epsilon1, epsilon2)

    # Define the loss function (mean squared error) and optimizer (SGD)
    criterion = nn.MSELoss()

    optimizer = optim.SGD(model.parameters(), lr = lr, momentum=0, weight_decay = wd)

    epoch_num = np.zeros((100,))
    loss_list = np.zeros((100,))
    ps_list = np.zeros((100,))
    ccgp_list = np.zeros((100,))
    pr_list = np.zeros((100, 2))
    pc_list = np.zeros((5, X_train0.size(0), X_train0.size(0)))
    s_list = np.zeros((5, X_train0.size(0)))
    h_list = np.zeros((5, hidden_size, X_train0.size(0)))

    # training
    idx = 0
    idx_s = 0
    for epoch in range(num_epochs):
        X_train = X_train0 + noise*(torch.rand(X_train0.size(0), input_size) - 0.5)
        optimizer.zero_grad()
        model.train()
        pred, r = model(X_train)
        loss = criterion(pred, y_train) #+0.15/400*torch.norm(x, 1)
        loss.backward()
        optimizer.step()

        if (epoch+1) % (num_epochs//100) == 0:
            epoch_num[idx] = epoch
            model.eval()
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            #r1 = torch.relu(torch.mm(w1, X_train.T) + biasl)
            r1 = r.detach().T
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            loss_list[idx] = loss.item()
            r1n = r1.numpy() - r1mean.numpy()
            c1 = np.matmul(r1n.T, r1n)
            ps_list[idx] = ps_highD(r1n.T, y_train.numpy())
            #ccgp_list[idx] = ccgp4(r1n)
            pr_list[idx, 0] = np.trace(c1)**2/np.linalg.norm(c1)**2
            idx = idx + 1

        if ((epoch+1) % (num_epochs//4) == 0) or (epoch == 0):
            model.eval()
            if print_idx == 1:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
            w1 = model.fc1.weight.data
            b1 = model.fc1.bias.data
            biasl = torch.mm(torch.unsqueeze(b1.T,1),torch.ones(1, X_train.size(0)))
            r1 = r.detach().T
            r1mean = torch.mm(torch.mean(r1,1,True), torch.ones(1, X_train.size(0),dtype = torch.float32))
            h_list[idx_s,:,:] = r1.numpy()
            idx_s = idx_s + 1

    return epoch_num, loss_list, ps_list, ccgp_list, pr_list, pc_list,  s_list, h_list